# 🎵 Ses İşareti Analizi ve Cinsiyet Sınıflandırma
### 2025-2026 Bahar Dönemi — Dönemiçi Proje
---

In [ ]:
# ============================================================
# 1. KÜTÜPHANELER
# ============================================================
import os
import glob
import numpy as np
import pandas as pd
import librosa
import librosa.display
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from IPython.display import display, Audio, HTML
import ipywidgets as widgets
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print("✅ Tüm kütüphaneler başarıyla yüklendi.")


## 📁 Adım 1: Veri Setini Yükleme ve Master Tablo Oluşturma

In [ ]:
# ============================================================
# 2. VERİ SETİ OKUMA
# ============================================================

DATASET_PATH = "Dataset/"   # <-- Kendi klasör yolunuzu buraya yazın

def load_master_dataframe(dataset_path):
    """Tüm grup Excel dosyalarını okuyup tek master DataFrame oluşturur."""
    excel_files = glob.glob(os.path.join(dataset_path, "**/Grup_*.xlsx"), recursive=True)
    
    if not excel_files:
        print("⚠️  Excel dosyası bulunamadı. DATASET_PATH değişkenini kontrol edin.")
        return pd.DataFrame()
    
    dfs = []
    for f in excel_files:
        try:
            df = pd.read_excel(f)
            df['Kaynak_Excel'] = os.path.basename(f)
            dfs.append(df)
            print(f"  ✔ {os.path.basename(f)} okundu — {len(df)} kayıt")
        except Exception as e:
            print(f"  ✖ {f} okunamadı: {e}")
    
    if not dfs:
        return pd.DataFrame()
    
    master = pd.concat(dfs, ignore_index=True)
    
    # Dosya varlığını kontrol et
    master['Dosya_Var'] = master['Dosya_Path'].apply(
        lambda p: os.path.exists(p) if isinstance(p, str) else False
    )
    
    print(f"\n📊 Toplam kayıt: {len(master)}")
    print(f"✅ Mevcut dosya: {master['Dosya_Var'].sum()}")
    print(f"❌ Eksik dosya : {(~master['Dosya_Var']).sum()}")
    return master

master_df = load_master_dataframe(DATASET_PATH)
display(master_df.head())


## 🔬 Adım 2: Sinyal İşleme Fonksiyonları

In [ ]:
# ============================================================
# 3. SİNYAL İŞLEME FONKSİYONLARI
# ============================================================

def compute_zcr(signal, frame_length=512, hop_length=256):
    """Sıfır Geçiş Oranı (ZCR) hesaplar."""
    zcr = librosa.feature.zero_crossing_rate(
        signal, frame_length=frame_length, hop_length=hop_length
    )
    return float(np.mean(zcr))


def compute_short_time_energy(signal, frame_length=512, hop_length=256):
    """Kısa Süreli Enerji (STE) hesaplar."""
    frames = librosa.util.frame(signal, frame_length=frame_length, hop_length=hop_length)
    energy = np.sum(frames ** 2, axis=0)
    return energy


def detect_voiced_frames(signal, sr, frame_length=512, hop_length=256,
                          energy_threshold_ratio=0.05, zcr_threshold=0.3):
    """
    Enerji + ZCR kullanarak sesli (voiced) çerçeveleri tespit eder.
    Döndürür: voiced boolian mask ve voiced sinyal parçası
    """
    energy = compute_short_time_energy(signal, frame_length, hop_length)
    zcr_frames = librosa.feature.zero_crossing_rate(
        signal, frame_length=frame_length, hop_length=hop_length
    )[0]
    
    # Eşik değerleri
    energy_thresh = energy_threshold_ratio * np.max(energy)
    
    voiced_mask = (energy > energy_thresh) & (zcr_frames < zcr_threshold)
    
    # Sesli çerçevelere karşılık gelen sinyal örneklerini topla
    voiced_signal = []
    for i, is_voiced in enumerate(voiced_mask):
        if is_voiced:
            start = i * hop_length
            end   = start + frame_length
            voiced_signal.append(signal[start:min(end, len(signal))])
    
    if voiced_signal:
        voiced_signal = np.concatenate(voiced_signal)
    else:
        voiced_signal = signal  # fallback
    
    return voiced_mask, voiced_signal


def compute_autocorrelation_f0(signal, sr, f0_min=50, f0_max=500):
    """
    Otokorelasyon yöntemiyle Temel Frekans (F0) hesaplar.
    R(τ) = Σ x[n] * x[n-τ]
    """
    # Otokorelasyon hesapla
    corr = np.correlate(signal, signal, mode='full')
    corr = corr[len(corr) // 2:]  # Sadece pozitif gecikmeler
    
    # F0 aralığına karşılık gelen lag sınırları
    lag_min = int(sr / f0_max)
    lag_max = int(sr / f0_min)
    lag_max = min(lag_max, len(corr) - 1)
    
    if lag_min >= lag_max:
        return None, corr
    
    # Bu aralıktaki maksimum tepe noktasını bul
    search_region = corr[lag_min:lag_max]
    peak_lag = np.argmax(search_region) + lag_min
    
    f0 = sr / peak_lag if peak_lag > 0 else None
    return f0, corr


def extract_features(file_path, sr_target=22050):
    """
    Bir ses dosyasından tüm özellikleri çıkarır.
    Döndürür: dict (f0, zcr, energy_mean, energy_std)
    """
    try:
        signal, sr = librosa.load(file_path, sr=sr_target)
        
        # Sesli bölgeleri tespit et
        _, voiced_signal = detect_voiced_frames(signal, sr)
        
        # Öznitelikler
        f0, _ = compute_autocorrelation_f0(voiced_signal, sr)
        zcr    = compute_zcr(voiced_signal)
        energy = compute_short_time_energy(voiced_signal)
        
        return {
            'f0'          : f0,
            'zcr'         : zcr,
            'energy_mean' : float(np.mean(energy)),
            'energy_std'  : float(np.std(energy)),
        }
    except Exception as e:
        print(f"  Hata [{file_path}]: {e}")
        return {'f0': None, 'zcr': None, 'energy_mean': None, 'energy_std': None}


print("✅ Sinyal işleme fonksiyonları tanımlandı.")


## 📊 Adım 3: Tüm Veri Seti İçin Öznitelik Çıkarımı

In [ ]:
# ============================================================
# 4. ÖZNİTELİK ÇIKARIMI DÖNGÜSÜ
# ============================================================

results = []

if master_df.empty:
    print("⚠️  Master DataFrame boş, örnek verilerle devam ediliyor...")
    # Demo: gerçek veri yoksa örnek satırlar
    demo_data = [
        {'Dosya_Path': 'demo_erkek.wav',  'Cinsiyet': 'Erkek', 'Yas': 35},
        {'Dosya_Path': 'demo_kadin.wav',  'Cinsiyet': 'Kadın', 'Yas': 28},
        {'Dosya_Path': 'demo_cocuk.wav',  'Cinsiyet': 'Çocuk', 'Yas': 8},
    ]
    working_df = pd.DataFrame(demo_data)
    working_df['Dosya_Var'] = False
else:
    working_df = master_df[master_df['Dosya_Var']].copy()

print(f"İşlenecek dosya sayısı: {len(working_df)}\n")

for idx, row in working_df.iterrows():
    print(f"  [{idx+1}/{len(working_df)}] {os.path.basename(str(row['Dosya_Path']))} işleniyor...", end=" ")
    features = extract_features(str(row['Dosya_Path']))
    features['Dosya_Path'] = row['Dosya_Path']
    features['Gercek_Etiket'] = row.get('Cinsiyet', 'Bilinmiyor')
    features['Yas']           = row.get('Yas', None)
    results.append(features)
    print(f"F0={features['f0']:.1f} Hz" if features['f0'] else "F0=Hesaplanamadı")

features_df = pd.DataFrame(results)
print("\n✅ Öznitelik çıkarımı tamamlandı!")
display(features_df)


## 📈 Adım 4: Otokorelasyon vs FFT Karşılaştırması

In [ ]:
# ============================================================
# 5. OTOKORELASYOn vs FFT GRAFİĞİ
# ============================================================

def plot_autocorr_vs_fft(file_path, sr_target=22050, title="Örnek Ses"):
    """Seçilen ses için otokorelasyon ve FFT grafiklerini yan yana çizer."""
    signal, sr = librosa.load(file_path, sr=sr_target)
    _, voiced = detect_voiced_frames(signal, sr)
    
    # Analiz için küçük bir pencere al (ilk 2048 örnek)
    window = voiced[:2048] if len(voiced) >= 2048 else voiced
    
    # Otokorelasyon
    f0_acorr, corr = compute_autocorrelation_f0(voiced, sr)
    
    # FFT
    fft_vals  = np.abs(np.fft.rfft(window, n=4096))
    fft_freqs = np.fft.rfftfreq(4096, d=1/sr)
    
    # F0 from FFT (50-500 Hz arası ilk tepe)
    mask = (fft_freqs >= 50) & (fft_freqs <= 500)
    if mask.any():
        f0_fft = fft_freqs[mask][np.argmax(fft_vals[mask])]
    else:
        f0_fft = None
    
    # Grafik
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    fig.suptitle(f"{title}", fontsize=14, fontweight='bold')
    
    # Sol: Otokorelasyon
    lag_min = int(sr / 500)
    lag_max = int(sr / 50)
    lags = np.arange(len(corr[:lag_max]))
    axes[0].plot(lags[lag_min:lag_max], corr[lag_min:lag_max], color='steelblue')
    if f0_acorr:
        peak_lag = int(sr / f0_acorr)
        axes[0].axvline(peak_lag, color='red', linestyle='--',
                        label=f'F0 = {f0_acorr:.1f} Hz')
    axes[0].set_title("Otokorelasyon Fonksiyonu R(τ)")
    axes[0].set_xlabel("Gecikme (τ) — örnek")
    axes[0].set_ylabel("R(τ)")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Sağ: FFT Spektrumu
    axes[1].plot(fft_freqs[fft_freqs <= 800], fft_vals[fft_freqs <= 800], color='darkorange')
    if f0_fft:
        axes[1].axvline(f0_fft, color='red', linestyle='--',
                        label=f'F0 = {f0_fft:.1f} Hz')
    axes[1].set_title("FFT Büyüklük Spektrumu |X(f)|")
    axes[1].set_xlabel("Frekans (Hz)")
    axes[1].set_ylabel("|X(f)|")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig("otokorelasyon_vs_fft.png", dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"\n📌 Otokorelasyon F0 : {f0_acorr:.2f} Hz" if f0_acorr else "Otokorelasyon F0: Hesaplanamadı")
    print(f"📌 FFT F0           : {f0_fft:.2f} Hz"   if f0_fft   else "FFT F0: Hesaplanamadı")

# --- Kullanım ---
# Gerçek dosya varsa:
# plot_autocorr_vs_fft("Dataset/Grup_01/ornek.wav", title="Erkek Sesi — Örnek")

# Demo sinyal ile test:
sr_demo = 22050
t = np.linspace(0, 1, sr_demo)
demo_signal = (0.5 * np.sin(2 * np.pi * 150 * t) +   # F0 = 150 Hz (erkek)
               0.3 * np.sin(2 * np.pi * 300 * t) +
               0.1 * np.random.randn(sr_demo))

import tempfile, soundfile as sf
with tempfile.NamedTemporaryFile(suffix='.wav', delete=False) as tmp:
    sf.write(tmp.name, demo_signal, sr_demo)
    plot_autocorr_vs_fft(tmp.name, title="Demo Ses (150 Hz Erkek Sesi Simülasyonu)")


## 🤖 Adım 5: Kural Tabanlı Sınıflandırıcı

In [ ]:
# ============================================================
# 6. KURAL TABANLI SINIFLANDIRICI
# ============================================================
# F0 referans aralıkları (literatür değerleri):
#   Erkek : 85  – 180 Hz
#   Kadın : 165 – 255 Hz
#   Çocuk : 250 – 400 Hz
# NOT: Bu eşikleri kendi veri setinizin istatistiklerine
#      göre Adım 6'dan sonra güncelleyin!

ERKEK_MAX = 180   # Hz — erkek/kadın sınırı
KADIN_MAX = 255   # Hz — kadın/çocuk sınırı

def rule_based_classify(f0):
    """
    F0 değerine göre kural tabanlı cinsiyet tahmini.
    Döndürür: 'Erkek', 'Kadın', 'Çocuk' veya 'Belirsiz'
    """
    if f0 is None or np.isnan(f0):
        return 'Belirsiz'
    elif f0 <= ERKEK_MAX:
        return 'Erkek'
    elif f0 <= KADIN_MAX:
        return 'Kadın'
    else:
        return 'Çocuk'


# Tahminleri uygula
if not features_df.empty and 'f0' in features_df.columns:
    features_df['Tahmin'] = features_df['f0'].apply(rule_based_classify)
    print("Sınıflandırma tamamlandı:")
    display(features_df[['Dosya_Path', 'Gercek_Etiket', 'f0', 'Tahmin']])
else:
    print("⚠️  Öznitelik tablosu boş, örnek verilerle devam ediliyor.")
    sample = pd.DataFrame({
        'Gercek_Etiket': ['Erkek','Erkek','Kadın','Kadın','Çocuk','Çocuk'],
        'f0':            [130, 160, 200, 230, 270, 310]
    })
    sample['Tahmin'] = sample['f0'].apply(rule_based_classify)
    features_df = sample
    display(features_df)

print(f"\nErkek eşiği  : F0 ≤ {ERKEK_MAX} Hz")
print(f"Kadın eşiği  : {ERKEK_MAX} < F0 ≤ {KADIN_MAX} Hz")
print(f"Çocuk eşiği  : F0 > {KADIN_MAX} Hz")


## 📋 Adım 6: İstatistik Tablosu ve Başarı Analizi

In [ ]:
# ============================================================
# 7. İSTATİSTİK TABLOSU
# ============================================================

def print_stats_table(df):
    """Erkek/Kadın/Çocuk için F0 istatistikleri ve başarı oranları."""
    siniflar = ['Erkek', 'Kadın', 'Çocuk']
    rows = []
    
    for sinif in siniflar:
        subset = df[df['Gercek_Etiket'] == sinif]
        if subset.empty:
            continue
        
        f0_vals = subset['f0'].dropna()
        correct  = (subset['Gercek_Etiket'] == subset['Tahmin']).sum()
        total    = len(subset)
        basari   = (correct / total * 100) if total > 0 else 0
        
        rows.append({
            'Sınıf'           : sinif,
            'Örnek Sayısı'    : total,
            'Ortalama F0 (Hz)': round(f0_vals.mean(), 2) if not f0_vals.empty else '-',
            'Std. Sapma'      : round(f0_vals.std(),  2) if not f0_vals.empty else '-',
            'Başarı (%)'      : f"{basari:.1f}%"
        })
    
    stats_df = pd.DataFrame(rows)
    
    # Genel doğruluk
    valid = df[df['Tahmin'] != 'Belirsiz']
    genel_basari = accuracy_score(valid['Gercek_Etiket'], valid['Tahmin']) * 100 if not valid.empty else 0
    
    print("=" * 65)
    print("          İSTATİSTİK TABLOSU")
    print("=" * 65)
    display(stats_df.set_index('Sınıf'))
    print(f"\n🎯 Genel Doğruluk (Accuracy): {genel_basari:.2f}%")
    
    return stats_df, genel_basari

stats_df, genel_basari = print_stats_table(features_df)


In [ ]:
# ============================================================
# 8. CONFUSION MATRIX (KARIŞIKLIK MATRİSİ)
# ============================================================

def plot_confusion_matrix(df):
    """Confusion matrix çizer ve classification report basar."""
    valid = df[df['Tahmin'] != 'Belirsiz']
    if valid.empty:
        print("⚠️  Geçerli tahmin yok.")
        return
    
    labels = sorted(valid['Gercek_Etiket'].unique())
    cm = confusion_matrix(valid['Gercek_Etiket'], valid['Tahmin'], labels=labels)
    
    fig, ax = plt.subplots(figsize=(7, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=labels, yticklabels=labels,
                linewidths=0.5, ax=ax)
    ax.set_xlabel("Tahmin Edilen", fontsize=12)
    ax.set_ylabel("Gerçek Etiket",  fontsize=12)
    ax.set_title("Karışıklık Matrisi (Confusion Matrix)", fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig("confusion_matrix.png", dpi=150, bbox_inches='tight')
    plt.show()
    
    print("\n📊 Sınıflandırma Raporu:")
    print(classification_report(valid['Gercek_Etiket'], valid['Tahmin'], labels=labels))

plot_confusion_matrix(features_df)


In [ ]:
# ============================================================
# 9. F0 DAĞILIM GRAFİĞİ (Sınıf bazında)
# ============================================================

def plot_f0_distribution(df):
    siniflar = ['Erkek', 'Kadın', 'Çocuk']
    renkler  = ['steelblue', 'salmon', 'mediumpurple']
    
    fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=False)
    fig.suptitle("Sınıf Bazında F0 Dağılımı", fontsize=14, fontweight='bold')
    
    for ax, sinif, renk in zip(axes, siniflar, renkler):
        subset = df[df['Gercek_Etiket'] == sinif]['f0'].dropna()
        if not subset.empty:
            ax.hist(subset, bins=15, color=renk, edgecolor='white', alpha=0.85)
            ax.axvline(subset.mean(), color='black', linestyle='--',
                       label=f'Ort: {subset.mean():.1f} Hz')
            ax.legend(fontsize=9)
        ax.set_title(sinif, fontsize=12)
        ax.set_xlabel("F0 (Hz)")
        ax.set_ylabel("Frekans")
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig("f0_dagilim.png", dpi=150, bbox_inches='tight')
    plt.show()

plot_f0_distribution(features_df)


## 🖥️ Adım 7: İnteraktif Arayüz — Tek Ses Dosyası Sınıflandırma

In [ ]:
# ============================================================
# 10. İNTERAKTİF ARAYZ — ipywidgets
# ============================================================

dosya_input = widgets.Text(
    value='',
    placeholder='Ses dosyasının tam yolunu yazın (örn: Dataset/Grup_01/ses.wav)',
    description='📂 Dosya:',
    layout=widgets.Layout(width='70%')
)

tahmin_btn  = widgets.Button(description='🎯 Sınıflandır', button_style='primary',
                              layout=widgets.Layout(width='150px'))
cikti_alani = widgets.Output()

def siniflandir(b):
    cikti_alani.clear_output()
    with cikti_alani:
        path = dosya_input.value.strip()
        if not path:
            print("⚠️  Lütfen bir dosya yolu girin.")
            return
        if not os.path.exists(path):
            print(f"❌ Dosya bulunamadı: {path}")
            return
        
        print(f"🔍 Analiz ediliyor: {os.path.basename(path)} ...")
        feats = extract_features(path)
        tahmin = rule_based_classify(feats['f0'])
        
        emoji = {'Erkek': '👨', 'Kadın': '👩', 'Çocuk': '🧒', 'Belirsiz': '❓'}
        
        print("─" * 45)
        print(f"  F0 (Temel Frekans) : {feats['f0']:.2f} Hz" if feats['f0'] else "  F0: Hesaplanamadı")
        print(f"  ZCR (Sıfır Geçiş) : {feats['zcr']:.4f}"   if feats['zcr'] else "  ZCR: Hesaplanamadı")
        print(f"  Enerji Ort.        : {feats['energy_mean']:.4f}")
        print("─" * 45)
        print(f"  🏷️  TAHMİN  →  {emoji.get(tahmin, '❓')}  {tahmin}")
        print("─" * 45)
        print(f"\n  📊 Tüm Veri Seti Genel Doğruluğu: {genel_basari:.2f}%")
        
        # Sesi oynat
        display(Audio(path))

tahmin_btn.on_click(siniflandir)

print("=" * 55)
print("  🎙️  SES İŞARETİ CİNSİYET SINIFLANDIRMA ARACI")
print("=" * 55)
display(widgets.VBox([dosya_input, tahmin_btn, cikti_alani]))


## 🔍 Adım 8: Hata Analizi

In [ ]:
# ============================================================
# 11. HATA ANALİZİ
# ============================================================

def hata_analizi(df):
    """Yanlış sınıflandırılan dosyaları listeler ve açıklar."""
    yanlis = df[
        (df['Gercek_Etiket'] != df['Tahmin']) & 
        (df['Tahmin'] != 'Belirsiz')
    ].copy()
    
    if yanlis.empty:
        print("🎉 Hiç yanlış sınıflandırma yok!")
        return
    
    print(f"⚠️  Toplam {len(yanlis)} yanlış tahmin:\n")
    
    for _, row in yanlis.iterrows():
        dosya = os.path.basename(str(row.get('Dosya_Path', 'demo')))
        gercek = row['Gercek_Etiket']
        tahmin = row['Tahmin']
        f0_val = f"{row['f0']:.1f} Hz" if row['f0'] else "Bilinmiyor"
        
        # Olası neden tahmini
        if gercek == 'Kadın' and tahmin == 'Çocuk':
            neden = "Yüksek perdeli kadın sesi (ince ses tonu) çocuk ile karışmış olabilir."
        elif gercek == 'Erkek' and tahmin == 'Kadın':
            neden = "Genç veya ince sesli erkek, kadın eşiğine yakın F0 üretmiş olabilir."
        elif gercek == 'Çocuk' and tahmin == 'Kadın':
            neden = "Ergenlik öncesi çocuk sesi kadın frekans aralığına girmiş olabilir."
        elif gercek == 'Erkek' and tahmin == 'Çocuk':
            neden = "Gürültülü kayıt veya duygusal konuşma F0 tespitini bozmuş olabilir."
        else:
            neden = "F0 eşik değeri, bu örnek için uygun olmayabilir."
        
        print(f"  📁 {dosya}")
        print(f"     Gerçek: {gercek}  →  Tahmin: {tahmin}  (F0: {f0_val})")
        print(f"     🔎 Olası neden: {neden}")
        print()

hata_analizi(features_df)


## ✅ Proje Özeti

| Bölüm | Durum |
|-------|-------|
| Veri yükleme & Master tablo | ✔ |
| ZCR + STE ile sesli bölge tespiti | ✔ |
| Otokorelasyon ile F0 hesabı | ✔ |
| Otokorelasyon vs FFT karşılaştırması | ✔ |
| Kural tabanlı sınıflandırıcı | ✔ |
| İstatistik tablosu | ✔ |
| Confusion Matrix | ✔ |
| F0 dağılım grafikleri | ✔ |
| İnteraktif arayüz | ✔ |
| Hata analizi | ✔ |

> **Not:** `DATASET_PATH` değişkenini ve `ERKEK_MAX` / `KADIN_MAX` eşiklerini kendi veri setinizin istatistiklerine göre güncelleyin.
